# Understanding RGB-IMU classification

Synthetic examples only: no training, weight download or accuracy measurement. This updated teaching notebook explains both full-class encoder families. The filename is retained for existing links; the legacy eight-action API is documented separately.

In [1]:
import numpy as np
import torch
from ai_research_work.har import SensorCNN, HOFGRU, HOFTransformer, fuse_probabilities
from ai_research_work.har.preprocessing import hof
torch.set_num_threads(1)
_ = torch.manual_seed(17)

## Shapes and classifier sizes

Sensor -> CNN. RGB -> optical flow -> HOF -> GRU or tiny Transformer. Random initialization demonstrates interfaces only. Synthetic sensor tensors below represent already-preprocessed inputs; real inference uses the local bundle's training-only scaler.

In [2]:
sensor = SensorCNN(classes=27).eval()
videos = {'HOF-GRU': HOFGRU(classes=27).eval(), 'HOF-Transformer': HOFTransformer(classes=27).eval()}
with torch.inference_mode():
    print('Sensor logits:', tuple(sensor(torch.zeros(2,128,6)).shape))
    for name, video in videos.items():
        print(name, 'logits:', tuple(video(torch.zeros(2,16,128)).shape), 'embedding:', tuple(video.embed(torch.zeros(2,16,128)).shape))
print('Sensor parameters:', sum(p.numel() for p in sensor.parameters()))
for name, video in videos.items():
    print(name, 'parameters:', sum(p.numel() for p in video.parameters()))

Sensor logits: (2, 27)
HOF-GRU logits: (2, 27) embedding: (2, 64)
HOF-Transformer logits: (2, 27) embedding: (2, 64)
Sensor parameters: 29883
HOF-GRU parameters: 39259
HOF-Transformer parameters: 40763


## Motion representation

Static synthetic RGB frames produce zero flow. A valid zero-motion observation is not the same as a missing modality.

In [3]:
features = hof(np.zeros((2,120,160,3), dtype=np.uint8))
print('HOF shape:', features.shape, 'total:', float(features.sum()))

HOF shape: (16, 128) total: 0.0


## Two probability rules and missingness

These three-class distributions are invented arithmetic examples, not measured activity predictions. Ordered class IDs and labels are explicit.

In [4]:
def record(p):
    return {'action_ids':[1,2,3], 'labels':['demo A','demo B','demo C'], 'probabilities':np.asarray([p],dtype=float)}
s = record([.7,.2,.1]); v = record([.4,.4,.2])
for rule in ['arithmetic','product']:
    result = fuse_probabilities(s,v,fusion=rule)
    print(rule, result['probabilities'].round(4).tolist(), result['status'])
    assert np.array_equal(fuse_probabilities(s,None,fusion=rule)['probabilities'],s['probabilities'])
    assert np.array_equal(fuse_probabilities(None,v,fusion=rule)['probabilities'],v['probabilities'])
print('Neither:', fuse_probabilities()['status'])

arithmetic [[0.55, 0.3, 0.15]] prediction_only
product [[0.7368, 0.2105, 0.0526]] prediction_only
Neither: insufficient_input


## Classification is not acceptance

A high maximum probability is not evidence that a novel activity is safe or known. The full-class API reports prediction_only and acceptance not_evaluated. No compatible operational rejection policy is shipped. Historical eight-action thresholds must not be applied to these classifiers. See [protocols](../docs/benchmark_protocols.md) and the [comparison notebook](03_compare_models.ipynb).